<a href="https://colab.research.google.com/github/jgromero/drl-csic/blob/main/code/from-scratch/cliffwalking-dqn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Q-Learning for _cliff walking_

[**Juan Gómez Romero**](https://ccia.ugr.es/~jgomez)  
Departamento de Ciencias de la Computación e Inteligencia Artificial  
Universidad de Granada  
This work is licensed under the [Apache License 2.0](https://choosealicense.com/licenses/apache-2.0/).

---
Based on:
> R.S. Sutton, A.G. Barto (2018) Reinforcement Learning. Chapters 6.5: Q-learning: Off-policy TD Control, 6.6: Expected Sarsa.

> Udacity, Deep Reinforcement Learning Course. Available on [GitHub](https://github.com/udacity/deep-reinforcement-learning/tree/master/dqn).

## Configuration

In [ ]:
%pip install "gymnasium>=1.1"

In [ ]:
# set render to 'ansi' (text output; change to 'human' for local display-enabled runtime)
render_mode = "ansi"

In [ ]:
# check whether supporting files exist
import os.path
if not os.path.isfile('dqn_agent.py'):
  !git clone https://github.com/jgromero/drl-csic
  %cd drl-csic/code/from-scratch

## Environment

We will be using [CliffWalking-v1](https://gymnasium.farama.org/environments/toy_text/cliff_walking/), the same environment that we solved with tabular Q-Learning in [cliffwalking-qlearning.ipynb](https://github.com/jgromero/drl-csic/blob/main/code/from-scratch/cliffwalking-qlearning.ipynb).

_Consider the gridworld shown to the right. This is a standard undiscounted, episodic task, with start and goal states, and the usual actions causing movement up, down, right, and left. Reward is -1 on all transitions except those into the region marked "The Cliff". Stepping into this region incurs a reward of -100 and sends the agent instantly back to the start._

<img src="https://github.com/jgromero/drl-csic/blob/main/code/cliffwalking.png?raw=true" width=500/>

We want to reuse, without any change, the same DQN agent that we used for [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/). This requires two adaptations of the environment:

*   The observation of CliffWalking is the index of the cell where the agent is (an integer from $0$ to $47$), but the Q-network expects a vector of numbers. We transform each observation into a _one-hot_ vector of size $48$ (all zeros, except a $1$ in the position of the cell) with the [`TransformObservation`](https://gymnasium.farama.org/api/wrappers/observation_wrappers/#gymnasium.wrappers.TransformObservation) wrapper.
*   Falling into the cliff does not end the episode, so a bad policy can walk forever. We limit the length of the episodes to $100$ steps with the `max_episode_steps` parameter.

The action and observation spaces can be printed as follows.

In [ ]:
import gymnasium as gym
import numpy as np
from gymnasium.wrappers import TransformObservation

def one_hot(state):
    """Encode the index of a cell (0-47) as a one-hot vector of size 48."""
    return np.eye(48, dtype=np.float32)[state]

env = gym.make("CliffWalking-v1", render_mode = render_mode, max_episode_steps=100)
env = TransformObservation(env, one_hot, gym.spaces.Box(0, 1, (48,), np.float32))
print(env.observation_space)
print(env.action_space)

We can interact with the environment with an untrained agent by using the `Agent` class in [dqn_agent.py](https://github.com/jgromero/drl-csic/blob/main/code/from-scratch/dqn_agent.py). The only difference with CartPole is the size of the state ($48$) and the number of actions ($4$).

In [ ]:
from dqn_agent import Agent

agent_untrained = Agent(state_size=48, action_size=4, seed=0)

state, _ = env.reset()

for j in range(100):
    action = agent_untrained.act(state)
    env.render()
    state, reward, terminated, truncated, _ = env.step(action)
    if terminated or truncated:
        break

env.close()

## DQN algorithm
Next we use the same implementation of Deep Q-Learning (DQN) as in [cartpole-dqn.ipynb](https://github.com/jgromero/drl-csic/blob/main/code/from-scratch/cartpole-dqn.ipynb). We rely on external files:

*   [dqn_agent.py](https://github.com/jgromero/drl-csic/blob/main/code/from-scratch/dqn_agent.py): Implementation of the agent capabilities (act, step, etc.)
*   [model.py](https://github.com/jgromero/drl-csic/blob/main/code/from-scratch/model.py): Implementation of the neural network of the agent (the Q-network)

The optimal policy (walking along the edge of the cliff) obtains a score of $-13$. During training, however, the agent keeps taking some random actions ($\epsilon$-greedy), which occasionally make it fall into the cliff. Therefore, we consider the problem solved when the agent obtains an average score of at least $-20$ points over 100 consecutive episodes.

In [ ]:
import random
import torch
import numpy as np
from collections import deque
from dqn_agent import device
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
SOLVED_SCORE = -20.0   # average score (last 100 episodes) to consider CliffWalking-v1 solved

def dqn(agent, n_episodes=2000, max_t=100, eps_start=1.0, eps_end=0.01, eps_decay=0.995, print_every=100):
    """Deep Q-Learning.

    Params
    ======
        agent (Agent): agent to train
        n_episodes (int): maximum number of episodes (n_episodes)
        max_t (int): maximum number of steps per episode (n_steps)
        eps_start (float): epsilon initial value
        eps_end (float): epsilon final value
        eps_decay (float): multiplication factor for epsilon (per episode)
        print_every (int): print score each print_every episodes
    """
    scores = []                        # scores for every episode
    scores_window = deque(maxlen=100)  # scores for the last 100 episodes
    eps = eps_start                    # initialize epsilon

    for i_episode in range(1, n_episodes+1):
        state, _ = env.reset()
        score = 0

        for t in range(max_t):

            # choose action At with e-greedy policy
            action = agent.act(state, eps)

            # apply At and get Rt+1, St+1
            next_state, reward, terminated, truncated, _ = env.step(action)

            # store <St, At, Rt+1, St+1>, train & update
            # (only 'terminated' marks St+1 as terminal; after a truncation we must still bootstrap)
            agent.step(state, action, reward, next_state, terminated)

            # advance to next state
            state = next_state
            score += reward

            if terminated or truncated:
                break

        scores_window.append(score)       # save last score (last 100)
        scores.append(score)              # save last score (every episode)
        eps = max(eps_end, eps_decay*eps) # reduce epsilon

        # print training info
        if i_episode % print_every == 0:
            print('Episode {}\tAverage score (last 100): {:.2f}'.format(i_episode, np.mean(scores_window)))
        if len(scores_window) == 100 and np.mean(scores_window) >= SOLVED_SCORE:
            print('\nProblem solved in {:d} episodes!\tAverage score (last 100): {:.2f}'.format(i_episode-100, np.mean(scores_window)))
            break
    else:
        print('\nProblem not solved in {:d} episodes.\tAverage score (last 100): {:.2f}'.format(n_episodes, np.mean(scores_window)))

    torch.save(agent.qnetwork_local.state_dict(), 'checkpoint-cliffwalking-dqn.pth') # save weights for trained agent

    return scores

agent = Agent(state_size=48, action_size=4, seed=0)
scores = dqn(agent)

# plot the scores
fig = plt.figure()
ax = fig.add_subplot(111)
plt.plot(np.arange(len(scores)), scores)
plt.ylabel('Score')
plt.xlabel('Episode #')
plt.show()

## Learned policy and state-value function

As in [cliffwalking-qlearning.ipynb](https://github.com/jgromero/drl-csic/blob/main/code/from-scratch/cliffwalking-qlearning.ipynb), we can print the resulting (greedy) policy. Here the values $Q(s, a)$ are not read from a table, but computed by the Q-network (`agent.qnetwork_local`) for the one-hot vectors of the 48 states.

In [ ]:
states = torch.eye(48).to(device)   # one-hot vectors of the 48 states
with torch.no_grad():
    Q = agent.qnetwork_local(states).cpu().numpy()   # Q values of the 48 states, shape (48, 4)

policy_plot = Q.argmax(axis=1).reshape(4, 12)
policy_plot[3, 1:] = -1   # cliff and goal cells are never visited
print("\nEstimated optimal policy (UP = 0, RIGHT = 1, DOWN = 2, LEFT = 3, N/A = -1):")
print(policy_plot)

We can visualize the values of the state-value function $v_\pi$ for the estimated policy $\pi$ and compare them with the optimal policy values $v_{\pi_*}$. Note that [dqn_agent.py](https://github.com/jgromero/drl-csic/blob/main/code/from-scratch/dqn_agent.py) uses $\gamma = 0.99$ instead of $\gamma = 1$, so the values of the states far from the goal are slightly higher (less negative) than the optimal ones.

(The `plot_values()` function is required for the visualization.)

In [ ]:
def plot_values(V):
    # reshape the state-value function
    V = np.reshape(V, (4,12))
    # plot the state-value function
    fig = plt.figure(figsize=(15,5))
    ax = fig.add_subplot(111)
    im = ax.imshow(V, cmap='cool')
    for (j,i),label in np.ndenumerate(V):
        ax.text(i, j, np.round(label,3), ha='center', va='center', fontsize=14)
    plt.tick_params(bottom=False, left=False, labelbottom=False, labelleft=False)
    plt.title('State-Value Function')
    plt.show()

__State-value function for the agent\'s learnt policy $v_\pi$__

In [ ]:
V = Q.max(axis=1)
V[37:] = 0   # cliff and goal cells are never visited
plot_values(V)

__State-value function for the optimal policy  $v_{\pi_*}$__

In [ ]:
# The state-values for the optimal policy can be manually calculated
V_opt = np.zeros((4,12))
V_opt[0][:] = -np.arange(3, 15)[::-1]
V_opt[1][:] = -np.arange(3, 15)[::-1] + 1
V_opt[2][:] = -np.arange(3, 15)[::-1] + 2
V_opt[3][0] = -13
plot_values(V_opt)

## Visualize trained agent

We can afterwards load the weights into the agent and visualize how it acts on the environment with the greedy policy. With `render_mode = "ansi"`, each step is printed as text (`x` is the agent, `C` the cliff and `T` the goal).

In [ ]:
# load weights from `checkpoint-cliffwalking-dqn.pth`
agent_load = Agent(state_size=48, action_size=4, seed=0)

agent_load.qnetwork_local.load_state_dict(torch.load('checkpoint-cliffwalking-dqn.pth', map_location=device, weights_only=True))

state, _ = env.reset()
score = 0
for j in range(100):
    action = agent_load.act(state)
    state, reward, terminated, truncated, _ = env.step(action)
    score += reward
    if render_mode == "ansi":
        print(env.render())
    if terminated or truncated:
        break
print('Score:', score)

env.close()